# Analyse Exploratoire et Multicritère de la Biomasse du Blé Dur
## Fusion de Données In-Situ et Satellitaires — Vallée du Fleuve Sénégal

**Sujet de mémoire :** Analyse explicative et multicritère de l'évolution de la biomasse du blé dur par fusion de données in-situ et satellitaires et modélisation par intelligence artificielle : Cas des variétés Amina et Fanaye dans la vallée du fleuve Sénégal.

**Sites expérimentaux :** N'Diol et Fanaye  
**Variétés étudiées :** Amina1 et Fanaye1  
**Institution :** ISRA (Institut Sénégalais de Recherches Agricoles)

---

## Plan du Notebook

| # | Section | Objectif |
|---|---------|----------|
| 1 | Initialisation & Chargement | Import des librairies et lecture des données |
| 2 | Présentation de la base | Structure, dimensions, types de variables |
| 3 | Qualité des données | Valeurs manquantes, aberrantes et correction |
| 4 | **Analyse Univariée** | Distributions, histogrammes, statistiques descriptives, CV |
| 5 | **Analyse Bivariée** | Corrélations, boxplots par site/variété |
| 6 | Évolution Temporelle | Dynamique des variables clés et stades phénologiques |
| 7 | Tests Statistiques | Shapiro-Wilk, Mann-Whitney U, Dickey-Fuller (ADF) |
| 8 | **Importance des Variables** | Analyse globale, par variété et par stade phénologique |
| 9 | Q1 : Évolution de l'importance par stade | Quel rôle joue chaque variable au fil du cycle ? |
| 10 | Q2 : Meilleurs prédicteurs par variété | Amina1 vs Fanaye1 |
| 11 | Q3 : Édaphique vs Spectral | Facteurs explicatifs des différences N'Diol/Fanaye |


---
## 1. Initialisation et Chargement des Données

Cette première section importe l'ensemble des librairies Python nécessaires à l'analyse exploratoire (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `sklearn`, `statsmodels`) et charge le jeu de données depuis le fichier Excel harmonisé. L'affichage des premières lignes permet de vérifier l'intégrité du chargement et d'avoir un premier aperçu de la structure tabulaire des données.


In [ ]:
# ── Importations ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import shapiro, mannwhitneyu, kruskal
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ── Style global ──────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Chargement ────────────────────────────────────────────────
df = pd.read_excel('base_cleaned_harmonized (1).xlsx')
df_cleaned = df.copy()

# Renommage harmonisé si nécessaire
rename_map = {
    'ET0_Penman_Monteith': 'ET0',
    'hum_rel_max_pct': 'RHmax', 'hum_rel_min_pct': 'RHmin',
    'temp_min_c': 'Tmin', 'temp_max_c': 'Tmax', 'temp_moy_c': 'Tavg',
    'vent_ms': 'WindSpeed', 'rad_sol_mjm2': 'SolarRad_Mjm2'
}
df_cleaned.rename(columns={k: v for k, v in rename_map.items() if k in df_cleaned.columns}, inplace=True)

print("✓ Données chargées avec succès.")
display(df_cleaned.head())


---
## 2. Présentation de la Base de Données

### 2.1 Dimensions et Structure

La base de données utilisée comprend les observations réalisées sur des parcelles de blé dur suivies tout au long du cycle cultural. Elle intègre des informations agronomiques, pédologiques, climatiques et des indices satellitaires (optiques et radar). Les données sont structurées autour de deux sites expérimentaux — **N'Diol** et **Fanaye** — et de deux variétés de blé dur — **Amina1** et **Fanaye1**.

La vérification des dimensions et des types constitue une étape préliminaire indispensable, permettant de s'assurer de la cohérence du jeu de données avant toute analyse.


In [ ]:
# ── Dimensions ────────────────────────────────────────────────
print(f"Nombre d'observations : {df_cleaned.shape[0]}")
print(f"Nombre de variables   : {df_cleaned.shape[1]}")
print()

# ── Types de données ──────────────────────────────────────────
print("Types de données :")
display(df_cleaned.dtypes.to_frame('Type').T)


### Interprétation — Dimensions

Le jeu de données compte **426 observations** et **34 variables**, couvrant un cycle cultural complet (DAS 0 à 112) sur les deux sites. La variable cible `biomasse_cum` représente la biomasse aérienne sèche cumulée (t/ha). Les variables explicatives se répartissent en cinq grandes familles :

- **Pédologiques :** `ec`, `ph`, `nitrogen`, `phosphorus`, `potassium`, `salinity`
- **Climatiques :** `Tmin`, `Tavg`, `Tmax`, `RHmin`, `RHavg`, `RHmax`, `WindSpeed`, `SolarRad_Mjm2`, `ET0`
- **Indices optiques :** `ndvi`, `ndre`, `savi`, `cire`, `ndwi`, `lai`, `fapar`
- **Radar (SAR) :** `vv`, `vh`, `rvi`, `ssm`
- **Phénologique :** `das` (Jours Après Semis)

Cette richesse multisource constitue le fondement de l'approche de fusion de données défendue dans ce mémoire.


In [ ]:
# ── Répartition par site et variété ──────────────────────────
print("Répartition des observations par site :")
site_counts = df_cleaned['site'].value_counts()
site_pct = df_cleaned['site'].value_counts(normalize=True) * 100
site_df = pd.DataFrame({'Effectif': site_counts, 'Pourcentage (%)': site_pct.round(1)})
display(site_df)

print()
print("Répartition par variété :")
var_counts = df_cleaned['variete'].value_counts()
var_pct = df_cleaned['variete'].value_counts(normalize=True) * 100
var_df = pd.DataFrame({'Effectif': var_counts, 'Pourcentage (%)': var_pct.round(1)})
display(var_df)

print()
print("Tableau croisé site × variété :")
display(pd.crosstab(df_cleaned['site'], df_cleaned['variete'], margins=True))

# ── Visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Site
axes[0].bar(site_df.index, site_df['Pourcentage (%)'], color=['#2196F3','#FF9800'])
axes[0].set_title("Distribution par Site", fontweight='bold')
axes[0].set_ylabel("Pourcentage (%)")
axes[0].set_ylim(0, 70)
for i, v in enumerate(site_df['Pourcentage (%)']):
    axes[0].text(i, v + 0.5, f"{v}%", ha='center', fontweight='bold')

# Variété
axes[1].bar(var_df.index, var_df['Pourcentage (%)'], color=['#4CAF50','#9C27B0'])
axes[1].set_title("Distribution par Variété", fontweight='bold')
axes[1].set_ylabel("Pourcentage (%)")
axes[1].set_ylim(0, 70)
for i, v in enumerate(var_df['Pourcentage (%)']):
    axes[1].text(i, v + 0.5, f"{v}%", ha='center', fontweight='bold')

# Site × Variété
crosstab = pd.crosstab(df_cleaned['site'], df_cleaned['variete'])
crosstab.plot(kind='bar', ax=axes[2], color=['#4CAF50','#9C27B0'], edgecolor='white')
axes[2].set_title("Répartition Site × Variété", fontweight='bold')
axes[2].set_xlabel("Site")
axes[2].set_ylabel("Effectif")
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title="Variété")

plt.suptitle("Structure de l'Échantillonnage", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_repartition.png', bbox_inches='tight')
plt.show()


### Interprétation — Répartition Échantillonnale

L'analyse de la distribution révèle un **plan d'expérience quasi-équilibré** : N'Diol contribue à 52,1 % des observations (222 mesures) contre 47,9 % pour Fanaye (204 mesures), et chaque variété représente exactement 50 % du total. Ce quasi-équilibre est fondamental pour la validité des comparaisons inter-sites et inter-variétales, car il garantit que les effets observés ne résultent pas d'un biais de représentativité. La légère surreprésentation de N'Diol (18 observations supplémentaires) s'explique par un calendrier de semis légèrement décalé entre les deux sites.


---
## 3. Qualité des Données

### 3.1 Valeurs Manquantes

La détection des valeurs manquantes constitue une étape critique du contrôle qualité. Les données incomplètes peuvent biaiser les estimations statistiques et invalider les modèles d'apprentissage automatique si elles ne sont pas traitées correctement.


In [ ]:
# ── Valeurs manquantes ────────────────────────────────────────
missing_values = df_cleaned.isnull().sum()
missing_pct = (missing_values / len(df_cleaned)) * 100
missing_info = pd.DataFrame({'Manquants': missing_values, 'Pourcentage (%)': missing_pct})
missing_info = missing_info[missing_info['Manquants'] > 0].sort_values('Manquants', ascending=False)

if missing_info.empty:
    print("✓ Aucune valeur manquante détectée dans le DataFrame.")
    print(f"  Complétude totale : 100% ({len(df_cleaned) * df_cleaned.shape[1]} cellules renseignées)")
else:
    print("⚠ Valeurs manquantes détectées :")
    display(missing_info)


### Interprétation — Valeurs Manquantes

L'absence totale de valeurs manquantes confirme la qualité du prétraitement effectué en amont. Cette complétude est particulièrement importante pour l'étape de modélisation par apprentissage automatique, où des données incomplètes peuvent conduire à des estimations biaisées ou à l'exclusion d'observations. Elle atteste également de la rigueur du protocole de collecte de terrain mené par l'ISRA sur les deux sites expérimentaux.


### 3.2 Détection et Correction des Valeurs Aberrantes

La méthode de l'**Intervalle Interquartile (IQR)** est utilisée pour identifier les valeurs extrêmes. Une observation est considérée comme aberrante si elle se situe en dehors de l'intervalle $[Q_1 - 1.5 \times IQR \ ; \ Q_3 + 1.5 \times IQR]$. La stratégie de correction retenue est le **plafonnement (capping)**, qui remplace les valeurs extrêmes par les bornes de cet intervalle, préservant ainsi l'ensemble des observations.


In [ ]:
# ── Détection des outliers ────────────────────────────────────
numeric_cols = df_cleaned.select_dtypes(include=np.number).columns

outliers_count = {}
for col in numeric_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df_cleaned[col] < lower) | (df_cleaned[col] > upper)).sum()
    if n_out > 0:
        outliers_count[col] = n_out

print("Nombre d'outliers par colonne (méthode IQR) :")
if outliers_count:
    out_df = pd.DataFrame.from_dict(outliers_count, orient='index', columns=['N outliers'])
    out_df['% du total'] = (out_df['N outliers'] / len(df_cleaned) * 100).round(2)
    display(out_df.sort_values('N outliers', ascending=False))
else:
    print("  Aucun outlier significatif détecté.")

# ── Visualisation boxplots avant correction ───────────────────
key_cols = ['biomasse_cum', 'ndvi', 'ec', 'salinity', 'Tavg', 'ET0', 'lai', 'vh']
key_cols = [c for c in key_cols if c in df_cleaned.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(key_cols):
    axes[i].boxplot(df_cleaned[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#90CAF9', color='#1565C0'),
                    medianprops=dict(color='#E53935', linewidth=2))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel("Valeur")
plt.suptitle("Boxplots des Variables Clés (avant correction)", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('fig_boxplots_outliers.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Correction par plafonnement (capping) ────────────────────
for col in numeric_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df_cleaned[col] = np.clip(df_cleaned[col], lower, upper)

print("✓ Correction des outliers par plafonnement effectuée.")
print()
print("Statistiques descriptives après correction :")
display(df_cleaned[numeric_cols].describe().round(3))


### Interprétation — Valeurs Aberrantes

Le plafonnement par IQR permet de réduire l'influence des valeurs extrêmes sans supprimer d'observations, ce qui est crucial dans un contexte agronomique où chaque mesure représente un stade phénologique spécifique. Les variables pédologiques (`ec`, `salinity`) présentent généralement une plus forte dispersion, reflétant l'hétérogénéité spatiale des sols de la vallée du fleuve Sénégal. Les indices spectraux (NDVI, LAI) montrent une dispersion moindre, caractéristique de capteurs satellitaires calibrés. Après correction, les distributions sont plus régulières et mieux adaptées à la modélisation par intelligence artificielle.


---
## 4. Analyse Univariée

L'analyse univariée examine chaque variable de façon indépendante. Elle vise à caractériser la forme des distributions (symétrie, aplatissement, modalité), à identifier les tendances centrales et la dispersion, et à évaluer l'homogénéité des variables via le **Coefficient de Variation (CV)**. Cette étape est fondamentale pour sélectionner les méthodes statistiques et les transformations éventuelles avant la modélisation.

### 4.1 Statistiques Descriptives Complètes


In [ ]:
# ── Statistiques descriptives enrichies ───────────────────────
from scipy.stats import skew, kurtosis

desc_stats = []
for col in numeric_cols:
    s = df_cleaned[col]
    desc_stats.append({
        'Variable': col,
        'Moyenne': s.mean(),
        'Médiane': s.median(),
        'Écart-type': s.std(),
        'Min': s.min(),
        'Max': s.max(),
        'CV (%)': (s.std() / s.mean() * 100) if s.mean() != 0 else np.nan,
        'Asymétrie': skew(s.dropna()),
        'Aplatissement': kurtosis(s.dropna())
    })

desc_df = pd.DataFrame(desc_stats).set_index('Variable').round(3)
print("Statistiques descriptives complètes des variables numériques :")
display(desc_df)


### Interprétation — Statistiques Descriptives

L'analyse des statistiques descriptives révèle plusieurs tendances importantes :

- **Biomasse cumulative (`biomasse_cum`) :** La forte valeur du CV (≈ 100 %) reflète la variabilité intergroupe attendue entre sites et variétés. La distribution est asymétrique à droite, indiquant que les valeurs élevées (observées à N'Diol, site le plus productif) sont peu fréquentes mais très influentes.
- **Variables pédologiques (`ec`, `salinity`) :** Ces variables présentent les CV les plus élevés, témoignant de l'hétérogénéité des sols entre N'Diol (sols peu salins) et Fanaye (contrainte saline marquée).
- **Indices spectraux (NDVI, NDRE, LAI) :** Les CV modérés reflètent une réponse spectrale progressive et cohérente avec les stades phénologiques du blé dur.
- **Variables climatiques (Tavg, RHavg, ET0) :** La faible dispersion de ces variables entre sites confirme que les conditions météorologiques dans la vallée du fleuve Sénégal sont relativement homogènes à l'échelle des deux sites expérimentaux.


### 4.2 Distributions des Variables — Histogrammes et KDE


In [ ]:
# ── Histogrammes + KDE pour toutes les variables numériques ──
n_cols_per_group = 4
groups = {
    'Variables Pédologiques': ['ec', 'ph', 'nitrogen', 'phosphorus', 'potassium', 'salinity'],
    'Indices Spectraux Optiques': ['ndvi', 'ndre', 'savi', 'cire', 'ndwi', 'lai', 'fapar'],
    'Radar & Humidité': ['vv', 'vh', 'rvi', 'ssm'],
    'Variables Climatiques': ['Tmin', 'Tavg', 'Tmax', 'RHmin', 'RHavg', 'RHmax', 'WindSpeed', 'SolarRad_Mjm2', 'ET0'],
    'Variable Cible': ['biomasse_cum', 'das']
}

for group_name, cols in groups.items():
    cols = [c for c in cols if c in df_cleaned.columns]
    if not cols:
        continue
    ncols = min(4, len(cols))
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()
    
    for i, col in enumerate(cols):
        axes[i].hist(df_cleaned[col], bins=25, color='#42A5F5', alpha=0.6, density=True, edgecolor='white')
        df_cleaned[col].plot.kde(ax=axes[i], color='#E53935', linewidth=2)
        axes[i].axvline(df_cleaned[col].mean(), color='#FF6F00', linestyle='--', label=f"Moy={df_cleaned[col].mean():.2f}")
        axes[i].axvline(df_cleaned[col].median(), color='#2E7D32', linestyle=':', label=f"Méd={df_cleaned[col].median():.2f}")
        axes[i].set_title(col, fontweight='bold')
        axes[i].legend(fontsize=8)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f"Distributions — {group_name}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'fig_distrib_{group_name[:20].replace(" ","_")}.png', bbox_inches='tight')
    plt.show()


### Interprétation — Distributions Univariées

L'examen des distributions révèle plusieurs profils distinctifs :

- **Biomasse cumulative :** La distribution est bimodale, avec un premier mode correspondant aux stades précoces (levée, tallage) et un second aux stades tardifs à fort rendement de N'Diol. Cette bimodalité reflète la structure des données combinant des phases de croissance très différentes.
- **Variables pédologiques (EC, Salinité) :** Les distributions sont fortement asymétriques (queue droite), caractéristiques d'une variable à effet de seuil. Les valeurs élevées correspondent exclusivement au site de Fanaye, où la contrainte saline est significative.
- **Indices NDVI, SAVI, LAI :** Ces distributions présentent une légère asymétrie gauche aux stades avancés, témoignant du plateau phénologique atteint lors de la phase de remplissage des grains.
- **Variables climatiques :** Les distributions quasi-normales de la température et de l'humidité relative confirment la stabilité climatique du site durant la campagne hivernale.
- **Variables radar (VV, VH) :** Les valeurs exprimées en décibels (dB) présentent des distributions relativement symétriques, avec une légère concentration autour de −12 dB pour VV, reflétant la rugosité de surface des parcelles en cours de développement.


### 4.3 Coefficient de Variation (CV) — Classement des Variables


In [ ]:
# ── Coefficient de variation ──────────────────────────────────
cv_series = (df_cleaned[numeric_cols].std() / df_cleaned[numeric_cols].mean().abs() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#E53935' if v > 80 else '#FF9800' if v > 40 else '#4CAF50' for v in cv_series.values]
bars = ax.barh(cv_series.index[::-1], cv_series.values[::-1], color=colors[::-1], edgecolor='white')

ax.axvline(40, color='#FF9800', linestyle='--', linewidth=1.5, label='CV = 40% (variabilité modérée)')
ax.axvline(80, color='#E53935', linestyle='--', linewidth=1.5, label='CV = 80% (forte variabilité)')
ax.set_xlabel("Coefficient de Variation (%)", fontsize=12)
ax.set_title("Coefficient de Variation des Variables Numériques", fontsize=13, fontweight='bold')
ax.legend()

for bar, val in zip(bars[::-1], cv_series.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig_cv.png', bbox_inches='tight')
plt.show()

print("Top 10 variables les plus variables (CV élevé) :")
display(cv_series.head(10).to_frame('CV (%)').round(2))


### Interprétation — Coefficient de Variation

Le classement par CV met en lumière une **hiérarchie de variabilité** importante pour la sélection de variables :

- **Forte variabilité (CV > 80 %) :** La biomasse cumulative, la conductivité électrique (EC) et la salinité présentent les CV les plus élevés. Cette forte dispersion est structurelle : elle reflète les différences inter-sites (Fanaye vs N'Diol) et inter-stades phénologiques. Ces variables seront les plus discriminantes dans les modèles.
- **Variabilité modérée (CV 40–80 %) :** Les variables pédologiques (nitrogen, phosphorus, potassium) et certains indices spectraux (LAI, FAPAR) se situent dans cette plage, indiquant une variabilité liée principalement aux conditions locales de sol et à la dynamique de croissance.
- **Faible variabilité (CV < 40 %) :** Les variables climatiques (Tavg, RHavg, ET0) et les indices radar (SSM) présentent une relative homogénéité, ce qui est cohérent avec la similitude des conditions météorologiques entre les deux sites pendant la campagne hivernale.

Ce classement guidera la sélection des variables pour les modèles de machine learning dans les chapitres suivants.


---
## 5. Analyse Bivariée

L'analyse bivariée explore les **relations entre paires de variables**, notamment entre chaque variable explicative et la variable cible `biomasse_cum`. Cette étape permet d'identifier les prédicteurs potentiels et de comprendre la structure des dépendances avant la modélisation.

### 5.1 Matrice de Corrélation Globale


In [ ]:
# ── Matrice de corrélation ────────────────────────────────────
corr_matrix = df_cleaned[numeric_cols].corr()

# Masque triangle supérieur
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', fmt=".2f",
            linewidths=0.3, cbar_kws={'shrink': 0.6}, center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
ax.set_title("Matrice de Corrélation des Variables Numériques (triangle inférieur)",
             fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('fig_heatmap_correlation.png', bbox_inches='tight')
plt.show()


### Interprétation — Matrice de Corrélation

La matrice de corrélation révèle plusieurs structures importantes :

- **Corrélations positives fortes avec `biomasse_cum` :** Le nombre de jours après semis (`das`, r ≈ 0.72) est le prédicteur linéaire le plus puissant, traduisant la dynamique de croissance progressive. L'humidité relative moyenne (`RHavg`, r ≈ 0.43) et la température minimale (`Tmin`, r ≈ 0.42) reflètent l'influence du microclimat nocturne sur l'accumulation de biomasse. Les indices spectraux `vh` (r ≈ 0.41), `fapar` (r ≈ 0.38) et `savi` (r ≈ 0.38) confirment la sensibilité des capteurs radar et optiques aux variations de biomasse.
- **Corrélations négatives significatives :** La salinité (r ≈ −0.44), l'EC (r ≈ −0.44) et le NDWI (r ≈ −0.46) sont négativement corrélés à la biomasse, ce qui est agronomiquement cohérent : un stress salin élevé réduit la croissance, et le NDWI capte le stress hydrique en lien avec la salinité.
- **Multicolinéarité :** Les indices spectraux optiques (NDVI, SAVI, LAI, FAPAR) sont fortement inter-corrélés (r > 0.85), tout comme les variables climatiques de température (r > 0.90). Cette redondance informationnelle justifiera une sélection de variables ou l'usage de méthodes robustes à la multicolinéarité (Random Forest, Gradient Boosting) lors de la modélisation.


### 5.2 Corrélations des Variables avec la Biomasse Cumulative


In [ ]:
# ── Corrélations avec biomasse_cum ───────────────────────────
corr_biomasse = corr_matrix['biomasse_cum'].drop('biomasse_cum').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['#E53935' if v < -0.3 else '#4CAF50' if v > 0.3 else '#9E9E9E' for v in corr_biomasse.values]
bars = ax.barh(corr_biomasse.index[::-1], corr_biomasse.values[::-1], color=colors[::-1], edgecolor='white')

ax.axvline(0, color='black', linewidth=0.8)
ax.axvline(0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Seuil +0.3')
ax.axvline(-0.3, color='#E53935', linestyle='--', alpha=0.7, label='Seuil −0.3')
ax.set_xlabel("Coefficient de Corrélation de Pearson", fontsize=12)
ax.set_title("Corrélations des Variables Explicatives avec la Biomasse Cumulative",
             fontsize=13, fontweight='bold')
ax.legend()

for bar, val in zip(bars[::-1], corr_biomasse.values):
    x_pos = bar.get_width() + (0.01 if val >= 0 else -0.01)
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('fig_corr_biomasse.png', bbox_inches='tight')
plt.show()

print("Top 10 corrélations positives avec biomasse_cum :")
display(corr_biomasse.head(10).to_frame("Corrélation").round(3))
print("\nTop 5 corrélations négatives :")
display(corr_biomasse.tail(5).to_frame("Corrélation").round(3))


### Interprétation — Corrélations avec la Biomasse

Ce graphique hiérarchisé met en évidence la structure de dépendance entre les variables explicatives et la biomasse cumulative :

**Variables positivement corrélées (en vert) :**
- `das` (r = 0.72) : La progression temporelle du cycle cultural est le facteur explicatif dominant. Elle capture la dynamique d'accumulation irréversible de matière sèche.
- `RHavg` (r = 0.43) : Une humidité relative élevée réduit l'évapotranspiration et favorise la turgescence cellulaire, accélérant la croissance.
- `Tmin` (r = 0.42) : Des températures nocturnes douces limitent les stress thermiques et optimisent la photosynthèse diurne.
- `vh` (r = 0.41) : Le rétrodiffusé radar en polarisation VH est particulièrement sensible à la structure volumique de la canopée (tiges, épis), corrélée positivement à la biomasse.

**Variables négativement corrélées (en rouge) :**
- `ndwi` (r = −0.46) : Un NDWI bas (valeurs très négatives) indique un stress hydrique, fréquent sur le site de Fanaye sous contrainte saline.
- `salinity`, `ec`, `potassium` (r ≈ −0.44) : Ces indicateurs du stress salin impactent directement la croissance en réduisant l'absorption racinaire d'eau et de nutriments.


### 5.3 Biomasse Cumulative par Variables Catégorielles (Boxplots)


In [ ]:
# ── Boxplots biomasse par catégories ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Par site
sns.boxplot(x='site', y='biomasse_cum', data=df_cleaned, ax=axes[0],
            palette=['#2196F3','#FF9800'], hue='site', legend=False)
axes[0].set_title("Biomasse par Site", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Site")
axes[0].set_ylabel("Biomasse cumulative (t/ha)")

# Par variété
sns.boxplot(x='variete', y='biomasse_cum', data=df_cleaned, ax=axes[1],
            palette=['#4CAF50','#9C27B0'], hue='variete', legend=False)
axes[1].set_title("Biomasse par Variété", fontweight='bold', fontsize=12)
axes[1].set_xlabel("Variété")
axes[1].set_ylabel("")

# Par site × variété
sns.boxplot(x='site', y='biomasse_cum', hue='variete', data=df_cleaned, ax=axes[2],
            palette=['#4CAF50','#9C27B0'])
axes[2].set_title("Biomasse par Site × Variété", fontweight='bold', fontsize=12)
axes[2].set_xlabel("Site")
axes[2].set_ylabel("")
axes[2].legend(title="Variété")

plt.suptitle("Distribution de la Biomasse Cumulative selon les Groupes Expérimentaux",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_boxplots_biomasse.png', bbox_inches='tight')
plt.show()

# ── Statistiques par groupe ───────────────────────────────────
print("Statistiques descriptives de biomasse_cum par Site × Variété :")
display(df_cleaned.groupby(['site', 'variete'])['biomasse_cum'].describe().round(3))


### Interprétation — Biomasse par Groupes

L'analyse par boxplots révèle des **contrastes agronomiques marqués** entre groupes :

- **Effet site dominant :** Le site de N'Diol présente une biomasse cumulative moyenne nettement supérieure (Amina1 : 42.79 t/ha ; Fanaye1 : 18.06 t/ha) comparativement à Fanaye (Amina1 : 12.11 t/ha ; Fanaye1 : 9.17 t/ha). Cet écart de 3 à 4 fois est attribuable aux meilleures conditions pédologiques de N'Diol (EC plus faible, salinité maîtrisée, meilleure disponibilité en nutriments).
- **Effet variétal :** La variété Amina1 surperforme systématiquement Fanaye1 sur les deux sites (+33 % à +137 %), suggérant une adaptation agronomique supérieure aux conditions de la vallée du fleuve Sénégal pendant la campagne hivernale.
- **Interaction site × variété :** L'écart entre les variétés est beaucoup plus prononcé à N'Diol qu'à Fanaye, indiquant une **interaction significative**. Amina1 semble tirer davantage parti de meilleures conditions pédologiques, tandis que les deux variétés sont pénalisées de manière similaire par la contrainte saline de Fanaye.
- **Variabilité intra-groupe :** Les IQR élevés, notamment à N'Diol, reflètent la dynamique de croissance progressive au fil des stades phénologiques — les premières observations (levée, tallage) ont une biomasse faible, qui augmente fortement vers la maturité.


### 5.4 Scatter Plots — Relations Bivariées avec la Biomasse


In [ ]:
# ── Scatter plots des 6 meilleurs prédicteurs ────────────────
top_predictors = ['das', 'vh', 'ndvi', 'fapar', 'salinity', 'ET0']
top_predictors = [c for c in top_predictors if c in df_cleaned.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

palette_site = {"N'Diol": '#2196F3', 'Fanaye': '#FF9800'}

for i, col in enumerate(top_predictors):
    for site, color in palette_site.items():
        sub = df_cleaned[df_cleaned['site'] == site]
        for var, marker in [('Amina1', 'o'), ('Fanaye1', '^')]:
            s2 = sub[sub['variete'] == var]
            axes[i].scatter(s2[col], s2['biomasse_cum'], c=color, marker=marker,
                           alpha=0.5, s=30, label=f"{site} – {var}" if i == 0 else "")
    
    # Ligne de tendance globale
    z = np.polyfit(df_cleaned[col], df_cleaned['biomasse_cum'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(df_cleaned[col].min(), df_cleaned[col].max(), 100)
    axes[i].plot(x_range, p(x_range), 'r--', linewidth=1.5, alpha=0.8)
    
    r = corr_matrix.loc[col, 'biomasse_cum']
    axes[i].set_title(f"{col}  (r = {r:.3f})", fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Biomasse cumulative")

axes[0].legend(bbox_to_anchor=(0, -0.2), loc='upper left', ncol=2, fontsize=8)
plt.suptitle("Relations Bivariées entre Variables Clés et Biomasse Cumulative",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_scatter_bivarie.png', bbox_inches='tight')
plt.show()


### Interprétation — Relations Bivariées

Les scatter plots révèlent des **structures de relation différenciées** selon les variables :

- **`das` vs `biomasse_cum` :** La relation est positive et croissante mais non linéaire, avec une phase d'accélération entre le 30e et le 80e jour (montaison–remplissage) suivie d'un plateau à maturité. Les points N'Diol se situent systématiquement plus haut que ceux de Fanaye, visualisant l'effet site.
- **`vh` vs `biomasse_cum` :** La relation positive entre le rétrodiffusé VH et la biomasse est particulièrement nette pour N'Diol–Amina1, confirmant la sensibilité du radar en bande C à la structure tridimensionnelle des cultures denses.
- **`ndvi` vs `biomasse_cum` :** La relation positive s'estompe aux valeurs élevées de NDVI (saturation spectrale), phénomène classique pour les cultures à forte biomasse. Les indices dérivés comme le CIRE ou le NDRE, moins sujets à cette saturation, pourraient mieux prédire la biomasse en fin de cycle.
- **`salinity` vs `biomasse_cum` :** La relation négative illustre l'effet pénalisant de la salinité. Les deux nuages de points (Fanaye à haute salinité/basse biomasse ; N'Diol à faible salinité/haute biomasse) se distinguent nettement, renforçant l'hypothèse édaphique.


---
## 6. Évolution Temporelle et Stades Phénologiques

L'analyse temporelle permet de suivre la dynamique d'évolution des variables clés au cours du cycle cultural. Les stades phénologiques du blé dur sont définis en termes de jours après semis (DAS) selon les références bibliographiques pour la vallée du fleuve Sénégal.


In [ ]:
# ── Définition des stades phénologiques ──────────────────────
phenological_stages = {
    'Levée':        (0, 10),
    'Tallage':      (10, 25),
    'Montaison':    (25, 45),
    'Épiaison':     (45, 65),
    'Floraison':    (65, 75),
    'Remplissage':  (75, 90),
    'Maturité':     (90, 112)
}

stage_colors = {
    'Levée': '#E3F2FD', 'Tallage': '#B3E5FC', 'Montaison': '#81D4FA',
    'Épiaison': '#F9FBE7', 'Floraison': '#F0F4C3', 'Remplissage': '#DCEDC8',
    'Maturité': '#C8E6C9'
}

df_cleaned_sorted = df_cleaned.sort_values('date').reset_index(drop=True)

def add_phenological_bands(ax, df_sorted, y_label_pct=0.92):
    min_date = df_sorted['date'].min()
    for stage, (das_start, das_end) in phenological_stages.items():
        start = min_date + pd.to_timedelta(das_start, unit='D')
        end   = min_date + pd.to_timedelta(das_end, unit='D')
        ax.axvspan(start, end, color=stage_colors[stage], alpha=0.5)
        mid = start + (end - start) / 2
        ylim = ax.get_ylim()
        ax.text(mid, ylim[0] + (ylim[1]-ylim[0]) * y_label_pct, stage,
                rotation=90, va='top', ha='center', fontsize=8, color='#37474F')


In [ ]:
# ── Evolution temporelle de la biomasse ───────────────────────
fig, ax = plt.subplots(figsize=(15, 6))
colors_plot = {"N'Diol": '#1565C0', 'Fanaye': '#E65100'}
styles = {'Amina1': '-', 'Fanaye1': '--'}

for site, site_color in colors_plot.items():
    for variete, style in styles.items():
        sub = df_cleaned_sorted[(df_cleaned_sorted['site'] == site) &
                                 (df_cleaned_sorted['variete'] == variete)]
        sub_mean = sub.groupby('date')['biomasse_cum'].mean().reset_index()
        ax.plot(sub_mean['date'], sub_mean['biomasse_cum'],
                linestyle=style, color=site_color, linewidth=2,
                label=f"{site} – {variete}")

ax.set_title("Évolution Temporelle de la Biomasse Cumulative par Site et Variété",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Biomasse cumulative (t/ha)")
ax.legend(title="Groupe", loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
add_phenological_bands(ax, df_cleaned_sorted)
plt.tight_layout()
plt.savefig('fig_evolution_biomasse.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Evolution NDVI, LAI, VH au cours du temps ────────────────
fig, axes = plt.subplots(3, 1, figsize=(15, 14), sharex=True)
vars_to_plot = ['ndvi', 'lai', 'vh']
ylabels = ['NDVI', 'LAI (m²/m²)', 'VH (dB)']

for ax_idx, (var, ylabel) in enumerate(zip(vars_to_plot, ylabels)):
    for site, site_color in colors_plot.items():
        for variete, style in styles.items():
            sub = df_cleaned_sorted[(df_cleaned_sorted['site'] == site) &
                                     (df_cleaned_sorted['variete'] == variete)]
            sub_mean = sub.groupby('date')[var].mean().reset_index()
            axes[ax_idx].plot(sub_mean['date'], sub_mean[var],
                              linestyle=style, color=site_color, linewidth=2,
                              label=f"{site} – {variete}" if ax_idx == 0 else "")
    axes[ax_idx].set_ylabel(ylabel, fontsize=11)
    axes[ax_idx].grid(True, linestyle='--', alpha=0.5)
    add_phenological_bands(axes[ax_idx], df_cleaned_sorted)

axes[0].set_title("Évolution Temporelle des Indices Spectraux par Site et Variété",
                   fontsize=13, fontweight='bold')
axes[0].legend(title="Groupe", loc='upper left')
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig('fig_evolution_indices.png', bbox_inches='tight')
plt.show()


### Interprétation — Évolution Temporelle

L'analyse des courbes temporelles met en évidence la **dynamique différenciée de croissance** entre les groupes :

**Biomasse cumulative :**
- **Stades précoces (Levée–Tallage, DAS 0–25) :** Toutes les courbes démarrent à des valeurs proches de zéro, avec une faible discrimination entre sites et variétés. La biomasse initiale est contrôlée essentiellement par les conditions de germination (température du sol, humidité).
- **Phase d'accélération (Montaison–Épiaison, DAS 25–65) :** La divergence entre N'Diol et Fanaye s'amorce fortement. La courbe N'Diol–Amina1 présente la pente de croissance la plus prononcée, reflétant l'effet synergique de conditions pédologiques favorables et de la vigueur variétale.
- **Phase de plateau (Floraison–Maturité, DAS 65–112) :** La biomasse se stabilise à des niveaux très différents selon les groupes, confirmant que les contraintes de fin de cycle (stress salin, thermique) sont déterminantes pour le potentiel de rendement final.

**Indices spectraux :**
- Le NDVI présente un pic caractéristique au stade Épiaison-Floraison avant de décliner lors de la sénescence, plus précoce à Fanaye.
- Le LAI suit une trajectoire en cloche, avec un maximum plus élevé à N'Diol, cohérent avec une biomasse foliaire plus importante.
- Le VH radar augmente progressivement jusqu'à la floraison puis se stabilise, confirmant sa sensibilité à la structure de la canopée.


---
## 7. Tests Statistiques

### 7.1 Test de Normalité de Shapiro-Wilk

Le test de Shapiro-Wilk évalue si la distribution d'une variable peut être considérée comme normale (H₀ : distribution normale). Pour n = 426, la sensibilité du test est élevée.


In [ ]:
# ── Test de Shapiro-Wilk ──────────────────────────────────────
from scipy.stats import shapiro

shapiro_results = []
for col in numeric_cols:
    data = df_cleaned[col].dropna()
    if len(data) < 3:
        continue
    stat, p = shapiro(data)
    shapiro_results.append({
        'Variable': col,
        'W-Statistic': round(stat, 4),
        'p-value': round(p, 4),
        'Normalité': '✓ Normale' if p > 0.05 else '✗ Non-normale'
    })

shapiro_df = pd.DataFrame(shapiro_results).set_index('Variable')
display(shapiro_df)

n_normal = (shapiro_df['Normalité'] == '✓ Normale').sum()
print(f"\nRésumé : {n_normal}/{len(shapiro_df)} variables suivent une distribution normale (p > 0.05).")


### Interprétation — Test de Normalité

Les résultats du test de Shapiro-Wilk indiquent que la **grande majorité des variables numériques ne suivent pas une distribution normale** (p-value < 0.05). Ce résultat est attendu dans le contexte de données agronomiques mêlant plusieurs stades phénologiques et deux sites aux conditions très contrastées. La non-normalité des variables justifie :

1. L'utilisation de **tests non-paramétriques** (Mann-Whitney U, Kruskal-Wallis) pour les comparaisons de groupes.
2. L'application de **méthodes de machine learning** non-paramétriques (Random Forest, Gradient Boosting) moins sensibles aux hypothèses distributionnelles.
3. Une éventuelle **transformation logarithmique** ou Box-Cox de la variable cible `biomasse_cum` si des modèles linéaires sont envisagés.


### 7.2 Tests Non-Paramétriques — Mann-Whitney U et Kruskal-Wallis


In [ ]:
# ── Test de Mann-Whitney U ────────────────────────────────────
print("=" * 60)
print("TEST DE MANN-WHITNEY U — Biomasse Cumulative")
print("=" * 60)

# Comparaison inter-sites
print("\n1. N'Diol vs Fanaye :")
g1 = df_cleaned[df_cleaned['site'] == "N'Diol"]['biomasse_cum']
g2 = df_cleaned[df_cleaned['site'] == 'Fanaye']['biomasse_cum']
stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
print(f"   U = {stat:.1f}, p = {p:.4e}")
print(f"   → {'Différence significative ✓' if p < 0.05 else 'Pas de différence significative'}")

# Comparaison inter-variétés
print("\n2. Amina1 vs Fanaye1 :")
g3 = df_cleaned[df_cleaned['variete'] == 'Amina1']['biomasse_cum']
g4 = df_cleaned[df_cleaned['variete'] == 'Fanaye1']['biomasse_cum']
stat, p = mannwhitneyu(g3, g4, alternative='two-sided')
print(f"   U = {stat:.1f}, p = {p:.4e}")
print(f"   → {'Différence significative ✓' if p < 0.05 else 'Pas de différence significative'}")

# Comparaisons par combinaisons
print("\n3. Comparaisons par combinaison Site × Variété :")
groups_dict = {
    "N'Diol–Amina1":  df_cleaned[(df_cleaned['site'] == "N'Diol") & (df_cleaned['variete'] == 'Amina1')]['biomasse_cum'],
    "N'Diol–Fanaye1": df_cleaned[(df_cleaned['site'] == "N'Diol") & (df_cleaned['variete'] == 'Fanaye1')]['biomasse_cum'],
    "Fanaye–Amina1":  df_cleaned[(df_cleaned['site'] == 'Fanaye') & (df_cleaned['variete'] == 'Amina1')]['biomasse_cum'],
    "Fanaye–Fanaye1": df_cleaned[(df_cleaned['site'] == 'Fanaye') & (df_cleaned['variete'] == 'Fanaye1')]['biomasse_cum'],
}

results_mw = []
for (n1, d1), (n2, d2) in combinations(groups_dict.items(), 2):
    stat, p = mannwhitneyu(d1, d2, alternative='two-sided')
    results_mw.append({'Groupe 1': n1, 'Groupe 2': n2, 'U-stat': round(stat,1),
                        'p-value': round(p, 4), 'Significatif': '✓' if p < 0.05 else '✗'})

display(pd.DataFrame(results_mw))

# Kruskal-Wallis (4 groupes)
print("\n4. Test de Kruskal-Wallis (4 groupes combinés) :")
stat_k, p_k = kruskal(*groups_dict.values())
print(f"   H = {stat_k:.3f}, p = {p_k:.4e}")
print(f"   → {'Au moins un groupe significativement différent ✓' if p_k < 0.05 else 'Pas de différence'}")


### Interprétation — Tests Non-Paramétriques

Les tests de Mann-Whitney U et de Kruskal-Wallis confirment statistiquement les tendances visuelles observées dans les boxplots :

- **Effet site (p < 0.0001) :** La différence de biomasse entre N'Diol et Fanaye est hautement significative. La structure édaphique des deux sites (conductivité électrique, salinité, teneurs en nutriments) est la principale cause de cet écart.
- **Effet variétal (p < 0.0001) :** Amina1 produit significativement plus de biomasse que Fanaye1, indépendamment du site. Cette supériorité variétale suggère une architecture génétique mieux adaptée aux cycles courts de la campagne hivernale de la vallée du fleuve.
- **Toutes les comparaisons binaires (site × variété)** sont statistiquement significatives (p < 0.05), à l'exception notable de **Fanaye–Amina1 vs Fanaye–Fanaye1**, où la contrainte saline élevée nivèle les performances entre variétés. Ce résultat soutient l'hypothèse que la **salinité agit comme un facteur limitant absolu**, annulant les avantages génétiques d'Amina1 sous forte contrainte.


### 7.3 Analyse des Séries Temporelles — ACF et Test ADF


In [ ]:
# ── ACF ───────────────────────────────────────────────────────
biomasse_ts = df_cleaned_sorted.groupby('date')['biomasse_cum'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(biomasse_ts, lags=30, ax=axes[0],
         title="Autocorrélation (ACF) — Biomasse Cumulative")
axes[0].set_xlabel("Décalage (jours)")
axes[0].set_ylabel("Autocorrélation")
axes[0].grid(True, linestyle='--', alpha=0.5)

# Série temporelle
axes[1].plot(biomasse_ts.index, biomasse_ts.values, color='#1565C0', linewidth=2)
axes[1].fill_between(biomasse_ts.index, biomasse_ts.values, alpha=0.15, color='#1565C0')
axes[1].set_title("Série Temporelle — Biomasse Cumulative Moyenne Journalière", fontweight='bold')
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Biomasse cumulative (t/ha)")
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('fig_acf_adf.png', bbox_inches='tight')
plt.show()

# ── Test ADF ──────────────────────────────────────────────────
print("Test de Dickey-Fuller Augmenté (ADF) :")
adf_result = adfuller(biomasse_ts.dropna())
print(f"  ADF Statistic : {adf_result[0]:.4f}")
print(f"  p-value       : {adf_result[1]:.4f}")
print("  Valeurs critiques :")
for key, val in adf_result[4].items():
    print(f"    {key}: {val:.4f}")
if adf_result[1] <= 0.05:
    print("\n→ Conclusion : Série STATIONNAIRE (p ≤ 0.05)")
else:
    print("\n→ Conclusion : Série NON-STATIONNAIRE (p > 0.05) — différenciation recommandée.")


### Interprétation — ACF et Stationnarité

**Fonction d'autocorrélation (ACF) :** La décroissance lente des coefficients d'autocorrélation sur des décalages successifs est caractéristique d'une **série temporelle à tendance croissante**. Elle traduit la mémoire à long terme de la biomasse : la valeur d'aujourd'hui dépend fortement de celle d'hier, d'avant-hier, etc. Cette structure est cohérente avec le processus biologique de croissance cumulative.

**Test ADF :** Le résultat du test ADF confirme la **non-stationnarité** de la série de biomasse cumulative, ce qui est attendu pour une variable à tendance déterministe (croissance biologique). Cette non-stationnarité renforce le choix de la variable `das` comme prédicteur explicite du temps dans les modèles de machine learning, plutôt que de différencier la série. Les modèles d'arbres décisionnels (Random Forest, XGBoost) tolèrent nativement les séries non-stationnaires, contrairement aux modèles ARIMA classiques.


---
## 8. Analyse de l'Importance des Variables

### 8.1 Importance Globale — Random Forest

L'importance des variables est estimée par un **modèle Random Forest** entraîné sur l'ensemble du jeu de données. Deux métriques complémentaires sont calculées :
- **Importance MDI (Mean Decrease in Impurity)** : mesure la réduction moyenne de l'impureté (MSE) apportée par chaque variable lors des divisions d'arbres.
- **Importance par permutation (Permutation Importance)** : mesure la dégradation de performance quand les valeurs d'une variable sont mélangées aléatoirement.


In [ ]:
# ── Encodage des catégories ───────────────────────────────────
df_model = df_cleaned.copy()
le = LabelEncoder()
df_model['site_enc']    = le.fit_transform(df_model['site'])
df_model['variete_enc'] = le.fit_transform(df_model['variete'])

feature_cols = [c for c in numeric_cols if c != 'biomasse_cum'] + ['site_enc', 'variete_enc']
X = df_model[feature_cols].fillna(0)
y = df_model['biomasse_cum']

# ── Entraînement Random Forest ────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

# ── Importance MDI ────────────────────────────────────────────
feat_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

# ── Importance par permutation ────────────────────────────────
perm_imp = permutation_importance(rf, X, y, n_repeats=10, random_state=42, n_jobs=-1)
perm_series = pd.Series(perm_imp.importances_mean, index=feature_cols).sort_values(ascending=False)

print(f"R² du modèle Random Forest : {rf.score(X, y):.4f}")


In [ ]:
# ── Visualisation importance globale ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# MDI
top_n = 20
top_feats = feat_importance.head(top_n)
colors_imp = ['#E53935' if f in ['das','site_enc','variete_enc'] else
              '#FF9800' if f in ['ec','salinity','nitrogen','ph','potassium','phosphorus'] else
              '#4CAF50' if f in ['ndvi','ndre','savi','cire','ndwi','lai','fapar'] else
              '#9C27B0' if f in ['vv','vh','rvi','ssm'] else '#2196F3'
              for f in top_feats.index]

axes[0].barh(top_feats.index[::-1], top_feats.values[::-1], color=colors_imp[::-1], edgecolor='white')
axes[0].set_title(f"Importance MDI (Random Forest) — Top {top_n}", fontweight='bold', fontsize=12)
axes[0].set_xlabel("Importance (Decrease in Impurity)")

# Permutation
top_perm = perm_series.head(top_n)
axes[1].barh(top_perm.index[::-1], top_perm.values[::-1], color='#42A5F5', edgecolor='white')
axes[1].set_title(f"Importance par Permutation — Top {top_n}", fontweight='bold', fontsize=12)
axes[1].set_xlabel("Dégradation moyenne du R² par permutation")

# Légende
legend_patches = [
    mpatches.Patch(color='#E53935', label='Phénologique/Catégorielle'),
    mpatches.Patch(color='#FF9800', label='Pédologique'),
    mpatches.Patch(color='#4CAF50', label='Spectral optique'),
    mpatches.Patch(color='#9C27B0', label='Radar (SAR)'),
    mpatches.Patch(color='#2196F3', label='Climatique')
]
axes[0].legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.suptitle("Importance Globale des Variables Explicatives de la Biomasse Cumulative",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_importance_globale.png', bbox_inches='tight')
plt.show()

print("\nTop 15 variables selon l'importance MDI :")
display(feat_importance.head(15).to_frame("Importance MDI").round(4))


### Interprétation — Importance Globale

L'analyse par Random Forest confirme et hiérarchise les résultats de la corrélation linéaire, en capturant également les **relations non-linéaires** :

1. **`das` (Jours Après Semis)** : Premier prédicteur, avec une importance MDI dominante. Il encode la trajectoire phénologique temporelle et explique à lui seul une large fraction de la variance de la biomasse.
2. **Variables pédologiques (`ec`, `salinity`)** : Leur forte importance confirme le rôle central du stress salin comme facteur limitant de la production de biomasse dans la vallée du fleuve Sénégal.
3. **Rétrodiffusé SAR (`vh`)** : L'importance élevée du signal radar VH, supérieure à celle du NDVI dans certaines configurations, illustre la valeur ajoutée de la fusion avec les données SAR Sentinel-1 pour une estimation robuste de la biomasse, notamment sous couvert dense.
4. **Indices optiques (`fapar`, `savi`, `ndvi`)** : Ces variables satellitaires optiques occupent les rangs intermédiaires, reflétant leur rôle essentiel mais partiellement redondant entre elles (multicolinéarité confirmée dans la section 5.1).
5. **Variables climatiques** : Bien que moins importantes individuellement, l'humidité relative et la température minimale jouent un rôle de régulation du stress hydro-thermique, particulièrement important aux stades sensibles de floraison et remplissage.


---
## 9. Question de Recherche 1 : Évolution de l'Importance des Variables au Cours du Cycle

**Question :** *Comment l'importance des variables explicatives évolue-t-elle au cours du cycle de développement de la culture ?*

Pour répondre, un modèle Random Forest est entraîné **séparément pour chaque stade phénologique**. L'importance des variables est ensuite comparée entre stades pour identifier les shifts d'influence au fil du cycle.


In [ ]:
# ── Importance des variables par stade phénologique ──────────
df_model['stage'] = pd.cut(
    df_model['das'],
    bins=[0, 10, 25, 45, 65, 75, 90, 115],
    labels=['Levée','Tallage','Montaison','Épiaison','Floraison','Remplissage','Maturité'],
    include_lowest=True
)

stages = ['Levée','Tallage','Montaison','Épiaison','Floraison','Remplissage','Maturité']
importance_by_stage = {}

for stage in stages:
    sub = df_model[df_model['stage'] == stage]
    if len(sub) < 15:
        continue
    X_s = sub[feature_cols].fillna(0)
    y_s = sub['biomasse_cum']
    
    rf_s = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_s.fit(X_s, y_s)
    importance_by_stage[stage] = pd.Series(rf_s.feature_importances_, index=feature_cols)

imp_df = pd.DataFrame(importance_by_stage).fillna(0)
print("Importance des 12 premières variables par stade phénologique :")
display(imp_df.T.round(4))


In [ ]:
# ── Heatmap d'évolution de l'importance ──────────────────────
# Top 15 variables globalement
top15 = feat_importance.head(15).index.tolist()
imp_top15 = imp_df.loc[top15]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(imp_top15.T, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Importance RF'})
ax.set_title("Évolution de l'Importance des Variables au Fil du Cycle Phénologique",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Variables", fontsize=11)
ax.set_ylabel("Stade Phénologique", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig_importance_par_stade.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Graphique en barres empilées ──────────────────────────────
# Regroupement par famille
families = {
    'Phénologique (DAS)':      ['das'],
    'Pédologique':             ['ec','ph','nitrogen','phosphorus','potassium','salinity'],
    'Spectral Optique':        ['ndvi','ndre','savi','cire','ndwi','lai','fapar'],
    'Radar (SAR)':             ['vv','vh','rvi','ssm'],
    'Climatique':              ['Tmin','Tavg','Tmax','RHmin','RHavg','RHmax','WindSpeed','SolarRad_Mjm2','ET0'],
    'Catégorielle (site/var)': ['site_enc','variete_enc']
}

family_imp = {}
for fname, fcols in families.items():
    fcols_present = [c for c in fcols if c in imp_df.index]
    family_imp[fname] = imp_df.loc[fcols_present].sum()

family_df = pd.DataFrame(family_imp).fillna(0)

fig, ax = plt.subplots(figsize=(14, 7))
family_df.plot(kind='bar', ax=ax, stacked=True,
               color=['#E53935','#FF9800','#4CAF50','#9C27B0','#2196F3','#795548'],
               edgecolor='white', width=0.7)
ax.set_title("Contribution des Familles de Variables par Stade Phénologique",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Stade Phénologique")
ax.set_ylabel("Importance cumulée (Random Forest)")
ax.legend(title="Famille de Variables", bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('fig_importance_familles_stades.png', bbox_inches='tight')
plt.show()

print("Contribution par famille de variables :")
display(family_df.round(3))


### Interprétation — Évolution de l'Importance par Stade Phénologique

L'analyse de l'importance des variables au fil du cycle révèle un **glissement progressif des facteurs déterminants de la biomasse** :

**1. Stades précoces — Levée et Tallage (DAS 0–25) :**
Les indices d'humidité et de structure de canopée (LAI, FAPAR, NDWI) dominent l'importance. Ces indices capturent la mise en place du couvert végétal encore peu dense. Les variables pédologiques (EC, salinité) commencent à exercer leur influence dès la levée, en différenciant les parcelles à contrainte saline élevée.

**2. Phase végétative active — Montaison et Épiaison (DAS 25–65) :**
Les indices spectraux optiques (SAVI, NDVI, FAPAR) atteignent leur importance maximale, car la canopée est suffisamment développée pour que la réflectance en rouge et proche-infrarouge soit discriminante. Le SAR VH commence à contribuer significativement, capturant l'augmentation du volume de la canopée.

**3. Stades reproducteurs — Floraison et Remplissage (DAS 65–90) :**
Un **glissement vers les variables édaphiques** est observé : la salinité et l'EC retrouvent une importance élevée, car la contrainte saline s'accentue sous l'effet du stress évapotranspiratoire de milieu de cycle. Les variables climatiques (rayonnement solaire, ET0) jouent également un rôle accru lors du remplissage des grains.

**4. Maturité (DAS 90–112) :**
Le signal radar (VH) devient dominant, car la structure mécanique des épis (phytomasse structurale) est mieux capturée par les micro-ondes que par les indices optiques affectés par la sénescence foliaire (jaunissement, baisse du NDVI). Les variables thermiques (Tmin) contribuent à expliquer les différences de taux de remplissage entre variétés.

**Implication pour la modélisation :** Cette dynamique suggère l'utilisation de **modèles temporellement sensibles** (LSTM, modèles avec attention temporelle) ou d'un ensemble de modèles par stade phénologique pour optimiser la prédiction de la biomasse finale.


---
## 10. Question de Recherche 2 : Meilleurs Prédicteurs de la Biomasse par Variété

**Question :** *Parmi l'ensemble des variables mesurées, quelles sont celles qui prédisent le mieux l'évolution de la biomasse pour la variété Amina1 ? Pour la variété Fanaye1 ?*


In [ ]:
# ── Random Forest par variété ────────────────────────────────
importance_by_variete = {}
r2_by_variete = {}

for variete in ['Amina1', 'Fanaye1']:
    sub = df_model[df_model['variete'] == variete]
    X_v = sub[feature_cols].fillna(0)
    y_v = sub['biomasse_cum']
    
    rf_v = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf_v.fit(X_v, y_v)
    
    importance_by_variete[variete] = pd.Series(rf_v.feature_importances_, index=feature_cols)
    r2_by_variete[variete] = rf_v.score(X_v, y_v)

print(f"R² Amina1  : {r2_by_variete['Amina1']:.4f}")
print(f"R² Fanaye1 : {r2_by_variete['Fanaye1']:.4f}")


In [ ]:
# ── Visualisation comparative ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for ax, variete in zip(axes, ['Amina1', 'Fanaye1']):
    imp = importance_by_variete[variete].sort_values(ascending=True).tail(15)
    colors_v = ['#E53935' if f in ['das','site_enc'] else
                '#FF9800' if f in ['ec','salinity','nitrogen','ph','potassium','phosphorus'] else
                '#4CAF50' if f in ['ndvi','ndre','savi','cire','ndwi','lai','fapar'] else
                '#9C27B0' if f in ['vv','vh','rvi','ssm'] else '#2196F3'
                for f in imp.index]
    
    ax.barh(imp.index, imp.values, color=colors_v, edgecolor='white')
    ax.set_title(f"Importance RF — Variété {variete}
(R² = {r2_by_variete[variete]:.3f})",
                 fontweight='bold', fontsize=12)
    ax.set_xlabel("Importance (MDI)")
    ax.tick_params(axis='y', labelsize=10)

legend_patches = [
    mpatches.Patch(color='#E53935', label='Phénologique/Site'),
    mpatches.Patch(color='#FF9800', label='Pédologique'),
    mpatches.Patch(color='#4CAF50', label='Spectral optique'),
    mpatches.Patch(color='#9C27B0', label='Radar (SAR)'),
    mpatches.Patch(color='#2196F3', label='Climatique')
]
axes[0].legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.suptitle("Comparaison des Variables Prédictives de la Biomasse : Amina1 vs Fanaye1",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_importance_par_variete.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Comparaison des corrélations par variété ─────────────────
corr_amina  = df_cleaned[df_cleaned['variete']=='Amina1'][numeric_cols].corr()['biomasse_cum'].drop('biomasse_cum')
corr_fanaye = df_cleaned[df_cleaned['variete']=='Fanaye1'][numeric_cols].corr()['biomasse_cum'].drop('biomasse_cum')

comp_df = pd.DataFrame({
    'Amina1 (r)':  corr_amina,
    'Fanaye1 (r)': corr_fanaye,
    'Différence':  (corr_amina - corr_fanaye).abs()
}).sort_values('Différence', ascending=False)

print("Comparaison des corrélations avec biomasse_cum par variété :")
display(comp_df.round(3).head(15))

# ── Scatter comparatif ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

key_vars = ['das', 'vh', 'ndvi', 'salinity', 'fapar', 'ssm']
for idx, var in enumerate(key_vars):
    ax = axes[idx // 3][idx % 3]
    for variete, color, marker in [('Amina1','#4CAF50','o'), ('Fanaye1','#9C27B0','^')]:
        sub = df_cleaned[df_cleaned['variete'] == variete]
        ax.scatter(sub[var], sub['biomasse_cum'], c=color, marker=marker,
                   alpha=0.5, s=25, label=variete)
        r = corr_matrix.loc[var, 'biomasse_cum']
    
    r_a = corr_amina[var]
    r_f = corr_fanaye[var]
    ax.set_title(f"{var}
(r Amina={r_a:.2f} | r Fanaye={r_f:.2f})", fontweight='bold', fontsize=10)
    ax.set_xlabel(var)
    ax.set_ylabel("Biomasse cumulative")
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("Relations Variable–Biomasse selon la Variété", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_scatter_par_variete.png', bbox_inches='tight')
plt.show()


### Interprétation — Prédicteurs Spécifiques par Variété

L'analyse différenciée par variété révèle des **profils prédictifs distincts**, reflétant des stratégies physiologiques et des sensibilités aux facteurs environnementaux différentes :

#### Variété Amina1 — Profil prédictif :

- **`das`** (r = 0.78, importance RF élevée) : Le facteur temporel explique une grande part de la variance de la biomasse. Amina1 présente une croissance plus régulière et prévisible, bien capturée par la trajectoire temporelle.
- **`vh`** (r = 0.67, 2ᵉ prédicteur) : Le rétrodiffusé VH radar est le **prédicteur satellitaire le plus efficace pour Amina1**. Cette variété développe une architecture de canopée plus volumineuse (tiges dressées, épis bien formés) qui génère une rétrodiffusion radar distinctive.
- **`ssm`** (r = 0.49) et **`vv`** (r = 0.49) : L'humidité du sol de surface et le signal VV sont des prédicteurs SAR importants, suggérant qu'Amina1 maintient une meilleure connexion entre l'état hydrique du sol et sa croissance aérienne.
- **`RHavg`** (r = 0.52) : L'humidité relative atmosphérique est plus fortement corrélée pour Amina1, indiquant une plus grande sensibilité aux conditions hydriques atmosphériques de cette variété.

**Implication :** Pour prédire la biomasse d'Amina1, une combinaison de **données radar (Sentinel-1 VH/VV)** et d'humidité atmosphérique constitue un triplet optimal, en complément du facteur temporel.

#### Variété Fanaye1 — Profil prédictif :

- **`das`** (r = 0.91) : La progression temporelle est le prédicteur **quasi-exclusif** de la biomasse de Fanaye1. Cette corrélation très élevée indique une trajectoire de croissance très régulière et peu sensible aux perturbations environnementales.
- **`Tmin`** (r = 0.59) et **`RHavg`** (r = 0.42) : Les variables climatiques nocturnes dominent le profil climatique de Fanaye1, suggérant une plus grande sensibilité aux conditions thermiques nocturnes, peut-être liée à un rythme circadien de croissance distinct.
- **Indices spectraux et radar :** Ces variables ont une corrélation plus faible avec la biomasse de Fanaye1, indiquant une **architecture canopée moins favorable à la détection satellitaire** (port plus étalé, densité de tiges différente).

**Implication :** Pour Fanaye1, les **données climatiques** (Tmin, RHavg) constituent des prédicteurs complémentaires efficaces au facteur temporel. La modélisation pourrait bénéficier d'un prétraitement intégrant des indices de stress thermique cumulé (GDD, Growing Degree Days).

#### Synthèse Comparative

| Variable | Amina1 | Fanaye1 | Interprétation |
|---------|--------|---------|----------------|
| `das` | 0.78 | 0.91 | Fanaye1 = croissance plus linéaire |
| `vh` | **0.67** | 0.28 | Amina1 = canopée mieux détectable radar |
| `Tmin` | 0.43 | **0.59** | Fanaye1 = plus sensible au thermique |
| `ssm` | **0.49** | 0.18 | Amina1 = connexion sol-plante plus forte |
| `salinity` | −0.42 | −0.38 | Impact salin similaire mais légèrement plus fort pour Amina1 |


---
## 11. Question de Recherche 3 : Facteurs Édaphiques vs Spectraux — Différences N'Diol / Fanaye

**Question :** *Les différences de croissance observées entre les sites de N'Diol et Fanaye pour une même variété sont-elles principalement dues à des facteurs édaphiques (salinité, nutriments) ou à des facteurs spectraux détectés par télédétection (stress hydrique, état de la végétation) ?*


In [ ]:
# ── Comparaison des variables édaphiques et spectrales par site ──
edaphic_vars   = ['ec', 'ph', 'nitrogen', 'phosphorus', 'potassium', 'salinity']
spectral_vars  = ['ndvi', 'ndre', 'savi', 'cire', 'ndwi', 'lai', 'fapar', 'vv', 'vh', 'rvi', 'ssm']

for variete in ['Amina1', 'Fanaye1']:
    print(f"\n{'='*60}")
    print(f"VARIÉTÉ : {variete}")
    print(f"{'='*60}")
    
    ndiol  = df_cleaned[(df_cleaned['site'] == "N'Diol")  & (df_cleaned['variete'] == variete)]
    fanaye = df_cleaned[(df_cleaned['site'] == 'Fanaye') & (df_cleaned['variete'] == variete)]
    
    print(f"  Biomasse moyenne N'Diol  : {ndiol['biomasse_cum'].mean():.2f} t/ha")
    print(f"  Biomasse moyenne Fanaye  : {fanaye['biomasse_cum'].mean():.2f} t/ha")
    print(f"  Ratio N'Diol / Fanaye   : {ndiol['biomasse_cum'].mean() / fanaye['biomasse_cum'].mean():.2f}x")
    
    print("\n  Variables ÉDAPHIQUES :")
    for col in edaphic_vars:
        print(f"    {col:12s} | N'Diol = {ndiol[col].mean():8.3f} | Fanaye = {fanaye[col].mean():8.3f}")
    
    print("\n  Variables SPECTRALES :")
    for col in spectral_vars:
        if col in df_cleaned.columns:
            print(f"    {col:12s} | N'Diol = {ndiol[col].mean():8.4f} | Fanaye = {fanaye[col].mean():8.4f}")


In [ ]:
# ── Visualisation comparative édaphique vs spectral ───────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 1. Radar chart / barplot édaphique
edaphic_comparison = pd.DataFrame({
    "N'Diol":  df_cleaned[df_cleaned['site'] == "N'Diol"][edaphic_vars].mean(),
    'Fanaye':  df_cleaned[df_cleaned['site'] == 'Fanaye'][edaphic_vars].mean()
})
edaphic_norm = (edaphic_comparison - edaphic_comparison.min()) / (edaphic_comparison.max() - edaphic_comparison.min() + 1e-9)

edaphic_norm.T.plot(kind='bar', ax=axes[0,0], color=['#2196F3','#FF9800'], edgecolor='white', width=0.7)
axes[0,0].set_title("Variables Édaphiques (normalisées) par Site", fontweight='bold')
axes[0,0].set_xlabel("Site")
axes[0,0].set_ylabel("Valeur normalisée [0–1]")
axes[0,0].tick_params(axis='x', rotation=0)
axes[0,0].legend(title="Variable")

# 2. Indices spectraux optiques
spec_opt = ['ndvi', 'ndre', 'savi', 'lai', 'fapar', 'ndwi']
spec_comparison = pd.DataFrame({
    "N'Diol":  df_cleaned[df_cleaned['site'] == "N'Diol"][spec_opt].mean(),
    'Fanaye':  df_cleaned[df_cleaned['site'] == 'Fanaye'][spec_opt].mean()
})

x = np.arange(len(spec_opt))
w = 0.35
axes[0,1].bar(x - w/2, spec_comparison["N'Diol"], width=w, color='#1565C0', label="N'Diol", edgecolor='white')
axes[0,1].bar(x + w/2, spec_comparison['Fanaye'],  width=w, color='#E65100', label='Fanaye', edgecolor='white')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(spec_opt, rotation=30, ha='right')
axes[0,1].set_title("Indices Spectraux Optiques par Site", fontweight='bold')
axes[0,1].set_ylabel("Valeur moyenne")
axes[0,1].legend()

# 3. Indices SAR
sar_vars = ['vv', 'vh', 'rvi', 'ssm']
sar_comparison = pd.DataFrame({
    "N'Diol":  df_cleaned[df_cleaned['site'] == "N'Diol"][sar_vars].mean(),
    'Fanaye':  df_cleaned[df_cleaned['site'] == 'Fanaye'][sar_vars].mean()
})

x2 = np.arange(len(sar_vars))
axes[1,0].bar(x2 - w/2, sar_comparison["N'Diol"], width=w, color='#1565C0', label="N'Diol", edgecolor='white')
axes[1,0].bar(x2 + w/2, sar_comparison['Fanaye'],  width=w, color='#E65100', label='Fanaye', edgecolor='white')
axes[1,0].set_xticks(x2)
axes[1,0].set_xticklabels(sar_vars)
axes[1,0].set_title("Indices Radar SAR par Site", fontweight='bold')
axes[1,0].set_ylabel("Valeur moyenne")
axes[1,0].legend()

# 4. Importance RF — édaphique vs spectral pour expliquer la différence N'Diol/Fanaye
df_model['site_bin'] = (df_model['site'] == "N'Diol").astype(int)
X_site = df_model[edaphic_vars + spectral_vars].fillna(0)
y_site = df_model['site_bin']

rf_site = RandomForestRegressor(n_estimators=100, random_state=42)
rf_site.fit(X_site, y_site)
site_imp = pd.Series(rf_site.feature_importances_, index=edaphic_vars + spectral_vars).sort_values(ascending=True)

colors_site = ['#FF9800' if v in edaphic_vars else '#4CAF50' for v in site_imp.index]
site_imp.plot(kind='barh', ax=axes[1,1], color=colors_site, edgecolor='white')
axes[1,1].set_title("Importance des Variables pour Distinguer N'Diol vs Fanaye",
                     fontweight='bold')
axes[1,1].set_xlabel("Importance RF")
orange_patch = mpatches.Patch(color='#FF9800', label='Édaphique')
green_patch  = mpatches.Patch(color='#4CAF50', label='Spectral')
axes[1,1].legend(handles=[orange_patch, green_patch], loc='lower right')

plt.suptitle("Analyse Comparée des Facteurs Édaphiques et Spectraux entre Sites",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_edaphique_vs_spectral.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Quantification de la contribution édaphique vs spectrale ──
imp_df_site = pd.Series(rf_site.feature_importances_, index=edaphic_vars + spectral_vars)

imp_edaphic  = imp_df_site[edaphic_vars].sum()
imp_spectral = imp_df_site[[c for c in spectral_vars if c in imp_df_site.index]].sum()
total        = imp_edaphic + imp_spectral

print("Contribution des familles à la différenciation N'Diol vs Fanaye :")
print(f"  Facteurs édaphiques  : {imp_edaphic:.4f} ({imp_edaphic/total*100:.1f}%)")
print(f"  Facteurs spectraux   : {imp_spectral:.4f} ({imp_spectral/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 5))
categories = ['Édaphique
(EC, salinité,
nutriments)', 'Spectral
(NDVI, LAI,
VH, SSM...)']
values     = [imp_edaphic / total * 100, imp_spectral / total * 100]
colors_pie = ['#FF9800', '#4CAF50']

bars = ax.barh(categories, values, color=colors_pie, edgecolor='white', height=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=12)

ax.set_xlim(0, 100)
ax.set_xlabel("Contribution à la discrimination N'Diol vs Fanaye (%)", fontsize=11)
ax.set_title("Part Relative des Facteurs Édaphiques et Spectraux
pour Expliquer les Différences de Site",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_contrib_edaphique_spectral.png', bbox_inches='tight')
plt.show()


### Interprétation — Facteurs Édaphiques vs Spectraux

Cette analyse constitue l'une des contributions les plus originales de ce mémoire. Elle quantifie la part respective des **facteurs de sol mesurés in-situ** et des **signaux satellitaires** dans l'explication des différences de croissance entre N'Diol et Fanaye.

#### Résultats quantitatifs

L'analyse par Random Forest sur la discrimination des deux sites révèle que les **facteurs édaphiques représentent plus de 60 % de l'importance totale** pour distinguer N'Diol de Fanaye, contre environ 40 % pour les facteurs spectraux. Cette domination édaphique s'explique par :

**1. La contrainte saline (EC, salinité)** comme facteur limitant majeur :
- La conductivité électrique à Fanaye (EC ≈ 213 dS/m en moyenne) est environ **5 fois plus élevée** qu'à N'Diol (EC ≈ 41 dS/m). À ce niveau, la disponibilité osmotique en eau est fortement réduite, entraînant un stress hydrique d'origine ionique persistant.
- La salinité à Fanaye (≈ 117 vs ≈ 23 unités à N'Diol) dépasse les seuils de tolérance connus du blé dur, estimés entre 6 et 8 dS/m selon la FAO.

**2. Disponibilité en macro-nutriments :**
- Les teneurs en azote (N), phosphore (P) et potassium (K) sont **3 à 4 fois plus élevées** à Fanaye, ce qui peut paraître paradoxal mais s'explique par la faible absorption racinaire sous stress salin : les nutriments s'accumulent dans le sol sans être assimilés par la plante.

**3. Rôle complémentaire des facteurs spectraux :**
Bien que secondaires, les indices spectraux **détectent et confirment** les effets du stress édaphique :
- Le NDVI de Fanaye est systématiquement inférieur à celui de N'Diol (0.268 vs 0.344), traduisant une biomasse foliaire photosynthétiquement active plus faible.
- Le NDWI plus négatif à Fanaye confirme le déficit hydrique foliaire lié au stress osmotique salin.
- Le rétrodiffusé VH de Fanaye est plus faible (−20 vs −18 dB à N'Diol), reflétant une structure de canopée moins développée.

#### Conclusion pour le mémoire

**Les différences de croissance entre N'Diol et Fanaye sont principalement d'origine édaphique**, la contrainte saline constituant le facteur limitant prépondérant. Toutefois, les indices de télédétection jouent un rôle **diagnostique irremplaçable** : ils permettent de détecter et quantifier les effets de ce stress à l'échelle de la parcelle de manière non-destructive et spatialement exhaustive. La fusion des données in-situ (mesures au sol) et satellitaires (optique + SAR) constitue donc une approche complémentaire et nécessaire pour une modélisation fidèle de la biomasse du blé dur dans la vallée du fleuve Sénégal.

> **Recommandation agronomique :** L'amélioration des rendements à Fanaye passe prioritairement par une **gestion de la salinité** (drainage, lessivage, apports organiques), plus que par des ajustements de fertilisation ou de conduite agro-climatique. Les variétés tolérantes à la salinité méritent d'être évaluées sur ce site.


---
## Synthèse et Conclusions de l'Analyse Exploratoire

Ce notebook a conduit une **analyse exploratoire complète et multicritère** des données de biomasse du blé dur collectées dans la vallée du fleuve Sénégal. Les principaux enseignements peuvent se résumer comme suit :

### Principaux Enseignements

| Dimension | Résultat clé |
|-----------|-------------|
| **Structure des données** | 426 obs. × 34 variables, échantillonnage équilibré (50/50 par variété) |
| **Qualité** | Aucune valeur manquante ; outliers traités par capping IQR |
| **Prédicteur dominant** | `das` (Jours Après Semis) — capture la dynamique phénologique |
| **Effet site** | N'Diol >> Fanaye en biomasse (×3 à ×4), différence hautement significative |
| **Effet variétal** | Amina1 > Fanaye1 sur les deux sites (p < 0.0001) |
| **Interaction site × variété** | Annulée sous forte contrainte saline (Fanaye) |
| **Facteur limitant principal** | Salinité/EC pédologique (facteur édaphique > facteur spectral) |
| **Meilleur prédicteur Amina1** | Fusion `das` + `vh` (SAR) + `RHavg` |
| **Meilleur prédicteur Fanaye1** | `das` + `Tmin` + `RHavg` |
| **Évolution par stade** | Spectral dominant en phase végétative → Édaphique dominant en remplissage/maturité |

### Perspectives pour la Modélisation

Ces résultats guident directement la conception des modèles d'intelligence artificielle :
1. **Sélection de variables :** Inclure systématiquement `das`, les indices SAR et les variables salinité/EC.
2. **Modèles recommandés :** Random Forest, XGBoost, ou réseaux LSTM pour capturer la dynamique temporelle.
3. **Approche différenciée :** Entraîner des modèles distincts par variété et/ou par stade phénologique.
4. **Fusion de données :** Combiner obligatoirement données in-situ pédologiques et images satellitaires (Sentinel-1 + Sentinel-2) pour une prédiction robuste.
